# Guidelines for data treatment with SCARLET

This notebook, present how to treat data with SCARLET python API

## 1. Workflow initialization: 

In [40]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory, WorkflowContext

RAW_DIR = "/home/achennev/python/scarlet/data/SANSLLB/raw_june" # change this to the path where your raw data is located
OUTPUT_DIR = "/home/achennev/python/scarlet/data/SANSLLB/out_june_v3" # change this to the path where you want to store the output data
INSTRUMENT_NAME = "sansllb" # instrument name, used to find the correct configuration files

RAW_DIR, OUTPUT_DIR


('/home/achennev/python/scarlet/data/SANSLLB/raw_june',
 '/home/achennev/python/scarlet/data/SANSLLB/out_june_v3')

Initialize the work flow context using the `initialize_workflow_context_from_raw_directory` function. This function perform several task : 

 - Read all the data files present in the raw data driectory, convert them into a nexus file with a generic file architecure and save them into the output_directory. 

 - Make a list of instrument configuration used during the experiment (collimation, detector distance, wavelength)

 - Detect if data file corresponds to "empty_cell", "dark", "water" or "empty_beam" measurements based on the sample name.


In [41]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    transmission_strategy="semi_transparent_beamstop",
    output_dir=OUTPUT_DIR,
    overwrite=True,
)
# If orverwrite is set to True, the output directory will be cleared before running the workflow. If set to False, the workflow will not run if the output directory already exists.

## 2. Save workflow and filter files
Sometimes, the raw_data folder contains many files that are not necessary for the data treatement and complexify the workflow operation. The 'WorkflowContext ' shall only contains runs intended to be treated and references (empty_cell,...).

For this prupose, a manual filtering or the edition of a csv file is possible.

First save the current state of the workflow in a csv file using the command:

In [42]:
w.write_runs_table_csv('runs_table.csv', overwrite=True)

PosixPath('/home/achennev/python/scarlet/notebooks/runs_table.csv')

After editing the csv file, you can run the workflow with the following code in order to load the filtered csv files. The runs can be displayed using the `w.runs_table()` command.

In [43]:
RUN_TABLE_CSV = 'runs_bs_semi.csv'
w.update_from_runs_table_csv(RUN_TABLE_CSV)
w.runs_table()


sample_name,config_id,mode,entity,thickness,transmission,file_path
EB,config_9,transmission,empty_beam,,,sans-llb2026n000975.nxs
Cd,config_9,scattering,dark,,,sans-llb2026n000982.nxs
EB,config_9,scattering,empty_beam,,,sans-llb2026n000983.nxs
AgBe,config_9,scattering,sample,1,0.770603,sans-llb2026n000984.nxs
EC,config_9,scattering,empty_cell,,0.948124,sans-llb2026n000985.nxs
H2O,config_9,scattering,water,1,0.525612,sans-llb2026n000986.nxs
ludox_AM20,config_9,scattering,sample,1,0.547647,sans-llb2026n000987.nxs
ludox_SM30,config_9,scattering,sample,1,0.551253,sans-llb2026n000988.nxs
ludox_TM50,config_9,scattering,sample,1,0.561255,sans-llb2026n000989.nxs
EB,config_10,transmission,empty_beam,,,sans-llb2026n000991.nxs


The entity column serves as a sample description. It can be set to :
- sample : normal sample
- water: water file allowing to compute flatfield
- empty_cell : empty_cell for substraction
- dark: B4C, Cd file for substraction
- empty_beam : note that a transmission empty_beam is required to compute the transmissions. 

The mode column can also be changed in case of miss recognition. 

The config_id column is just a simple identification string per confiruration used. The configurations parameters can be displayed using the `WorkflowContext.configurations_table()` method

In [45]:
w.configurations_table()

config_id,wavelength,sample_detector_distance,collimation_distance,last_aperture_to_sample_distance,aperture1,aperture2,notes
config_10,4.99998 A,detector0=2 m; detector1=0.809993 m; detector2=0.809993 m,3 m,1.6 m,slit x=0.06 m y=0.06 m,slit x=0.008 m y=0.008 m,
config_9,4.99998 A,detector0=7.99999 m; detector1=2.66667 m; detector2=2.66667 m,8 m,1.6 m,slit x=0.06 m y=0.06 m,slit x=0.008 m y=0.008 m,


The `WorkflowContext` can be saved a a nexus file for futur use using the following commands:

In [46]:

w.save("workflow.nxs", overwrite=True)
# load the workflow context from the saved file
w = WorkflowContext.load("workflow.nxs")

## 3. Transmission computation

The computation of the detector is performed using the detector named `detector0` in the nexus files refering to the central detector. During the initialization of the `WorkflowContext`, the ROI for transmission measurements is automatically detected.

The ROIs per config can be retrieved and modified using the following methods:

In [59]:
# read roi
print(w.get_roi('config_9'))
# set roi
# w.set_roi('config_10', [61, 65, 62, 64])  # Example ROI coordinates
# w.set_roi('config_9', [60, 64, 61, 63])  # Example ROI coordinates

[60, 64, 61, 63]


The computation of the transmission is perfomd using the `WorkflowContext.compute_transmissions()` method:

In [48]:
w.compute_transmissions()
# save transmission in csv file once computed
w.write_runs_table_csv("runs_bs_semi_2.csv",overwrite=True)
w.runs_table()

sample_name,config_id,mode,entity,thickness,transmission,file_path
EB,config_9,transmission,empty_beam,,,sans-llb2026n000975.nxs
Cd,config_9,scattering,dark,,,sans-llb2026n000982.nxs
EB,config_9,scattering,empty_beam,,,sans-llb2026n000983.nxs
AgBe,config_9,scattering,sample,1,0.755916,sans-llb2026n000984.nxs
EC,config_9,scattering,empty_cell,,0.925201,sans-llb2026n000985.nxs
H2O,config_9,scattering,water,1,0.507435,sans-llb2026n000986.nxs
ludox_AM20,config_9,scattering,sample,1,0.525092,sans-llb2026n000987.nxs
ludox_SM30,config_9,scattering,sample,1,0.52465,sans-llb2026n000988.nxs
ludox_TM50,config_9,scattering,sample,1,0.530498,sans-llb2026n000989.nxs
EB,config_10,transmission,empty_beam,,,sans-llb2026n000991.nxs


## 4. Masking


The edition of masks can be done using the `scarlet viewer` GUI. To run this graphical interface just run `scarlet viewer` command in a terminal with the appropriate virtual environnment activated. The mask files shall be saved in the output directory in order to be found by the workflow.

Each mask file stores the configuration over which they have been drawn. The `WorkflowContext` can automatically adress them to a specc

In [49]:
w.attach_mask_bundles_from_output_dir()
# display the mask files for a specific configuration
w.mask_files

{'config_9': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/mask_8m_viewer.nxs'),
 'config_10': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/masks_2m_viewer.nxs')}

Manual attibution of a mask file can be done using the following command:

In [51]:
# w.set_mask_file(config_id='config_10', file_path='/path/to/mask_file')

## 5. Flatfield / water normalization

The `water` entity in the run table can be used to generate flatfields files for normalization by water.
To build a flatfield file for a specific configuration use the following command:

In [52]:
w.build_water_flatfield(config_id="config_10")
w.build_water_flatfield(config_id="config_9")

PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/flatfield_config_9.nxs')

The flatflied file is saved in the output directory. It can also be set manually with:

In [53]:
# set flatfield file for a specific configuration
# w.set_flatfield(config_id="config_10", file_path="flat_field_path")
# get the flatfield file for a specific configuration
w.get_flatfield(config_id="config_9")

PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/flatfield_config_9.nxs')

If no water file is available at a given configuration we can declare to use the flatfield of one config to another one using the following command.

In [54]:
w.set_flatfield_source(config_id="config_9", source_config_id="config_10")
w.get_flatfield(config_id="config_9")

PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/flatfield_config_10.nxs')

## 6. Reduction pipeline

Once the workflow is set, the data reduction can be performed using a `Pipeline`. The default option perform the following reduction steps:
- Subtract dark and empty_cell
- Division by water flatfield               
- Azimutal averaging
- save processed data in the nexus files in the `processed` entry
- save the processed data as a .txt file for each configuration and each detector.

The reducrion can be perform for a given sample at a given configuration using:

In [55]:
from scarlet.workflow.pipeline import ReductionPipeline, ReductionState
pipeline = ReductionPipeline().default()
state = pipeline.run_for_sample(workflow=w, sample_name="ludox_SM30", config_id="config_9")

Or for the all workflow using:

In [56]:
pipeline.run_all(w)

In [57]:
# w.save("workflow.nxs")

w.load("workflow.nxs")

WorkflowContext(experiment_id='experiment', instrument_name='sansllb', root_dir=PosixPath('/home/achennev/python/scarlet/data/SANSLLB/raw_june'), output_dir=PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3'), runs={RunKey(config_id='config_9', entity='empty_beam', mode='transmission', sample_name='EB'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/sans-llb2026n000975.nxs'), RunKey(config_id='config_9', entity='dark', mode='scattering', sample_name='Cd'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/sans-llb2026n000982.nxs'), RunKey(config_id='config_9', entity='empty_beam', mode='scattering', sample_name='EB'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/sans-llb2026n000983.nxs'), RunKey(config_id='config_9', entity='sample', mode='scattering', sample_name='AgBe'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out_june_v3/sans-llb2026n000984.nxs'), RunKey(config_id='config_9', entity='empty_cell', 

The pipleline reduction steps can be enriched with other reduction scheme. If you have any suggestion, please contact me.

## 6. Concatenation of configurations

In [58]:
from scarlet.workflow.pipeline import StichingPipeline
stitch = StichingPipeline(w)
stitch.run_all(scale_on="config_10")